# Summit 2026 HOL — CoCo
## Module 07: Debug Queries, Validate Agent Behavior, and Optimize

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🛠️ Uses **CoCo** to debug an inefficient query generated by the agent
2. 🛠️ Shows how to optimize SQL performance using CoCo as a developer tool
3. 🛠️ Validates agent behavior by testing edge cases
4. 🔹 Introduces CoCo as the developer's companion for Snowflake AI workflows

---

### What is CoCo?

**CoCo** is an AI-driven developer agent integrated into Snowflake. It's available in two forms:
- **CoCo in Snowsight** — embedded in Workspaces and Notebooks for SQL/Python authoring
- **CoCo CLI** — a local terminal agent that bridges your dev environment (VS Code, Cursor) with Snowflake
- **CoCo Desktop** *(Launching at Summit 2026)* — native desktop app with full local file access, git integration, and the same agentic capabilities as the CLI in a visual interface

For this module, we'll use **CoCo in Snowsight** to debug and optimize queries that our Nexus Capital agent generates.

### When to Use CoCo vs. the Agent:

| | Cortex Agent (Modules 03–05) | CoCo (this module) |
|---|---|---|
| **Who uses it** | Business users, analysts | Developers, data engineers |
| **What it does** | Answers questions from data | Writes, debugs, and optimizes code |
| **Works with** | Semantic views, search services, MCP | SQL, Python, local files, git, shell |
| **Available in** | CoWork, REST API | Snowsight, CLI, Desktop *(new)* |
| **Billing** | AI credits (per token) | AI credits (per token) |

They're complementary: the agent serves users, CoCo serves builders.

> **Documentation:** [CoCo](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code) | [CoCo CLI](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code-cli)

---

### Estimated Time: **20–25 minutes**

### Prerequisites:
- Module 06 complete
- CoCo enabled on your account (it's GA for all commercial accounts with cross-region inference)

## Step 1: 🛠️ Open CoCo in Snowsight

### How to access:

1. In Snowsight, open any **Workspace** or **Notebook**
2. Look for the **CoCo** icon (sparkle/AI icon) in the editor
3. Or open the CoCo panel from the left sidebar

CoCo in Snowsight is context-aware — it knows which SQL file or notebook you're viewing and uses it as background context.

## Step 2: 🛠️ Debug an Inefficient Query

### Scenario: The agent generated a working but slow query

When you asked the agent "Which clients have the highest concentration in a single stock?", it generated SQL that works — but uses
correlated subqueries that scan the POSITIONS table multiple times.

On our small dataset this runs fast. On a production table with millions of rows, each additional scan multiplies execution time.
Let's prove it's inefficient, then use CoCo to fix it.

### First, run EXPLAIN on the inefficient query to see the execution plan:

In [ ]:
%%sql -r SetContext
USE ROLE NEXUS_ADMIN;
USE DATABASE NEXUS_HOL;
USE SCHEMA ANALYTICS;
USE WAREHOUSE NEXUS_WH;

In [ ]:
%%sql -r InefficientQuery
-- Run the inefficient query to confirm it returns correct results
-- Intentionally inefficient: calculates concentration using a subquery per row
SELECT
    c.CLIENT_NAME,
    p.SYMBOL,
    p.MARKET_VALUE,
    (SELECT SUM(p2.MARKET_VALUE) FROM NEXUS_HOL.ANALYTICS.POSITIONS p2 WHERE p2.CLIENT_ID = p.CLIENT_ID) AS TOTAL_PORTFOLIO,
    ROUND(p.MARKET_VALUE / (SELECT SUM(p2.MARKET_VALUE) FROM NEXUS_HOL.ANALYTICS.POSITIONS p2 WHERE p2.CLIENT_ID = p.CLIENT_ID) *
100, 2) AS CONCENTRATION_PCT
FROM NEXUS_HOL.ANALYTICS.POSITIONS p
JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON p.CLIENT_ID = c.CLIENT_ID
WHERE p.MARKET_VALUE / (SELECT SUM(p2.MARKET_VALUE) FROM NEXUS_HOL.ANALYTICS.POSITIONS p2 WHERE p2.CLIENT_ID = p.CLIENT_ID) > 0.10
ORDER BY CONCENTRATION_PCT DESC;

In [ ]:
%%sql -r InefficientQuery
-- Intentionally inefficient: calculates concentration using a subquery per row
-- Show the execution plan — look for multiple TableScan nodes on POSITIONS
EXPLAIN
SELECT
    c.CLIENT_NAME,
    p.SYMBOL,
    p.MARKET_VALUE,
    (SELECT SUM(p2.MARKET_VALUE) FROM NEXUS_HOL.ANALYTICS.POSITIONS p2 WHERE p2.CLIENT_ID = p.CLIENT_ID) AS TOTAL_PORTFOLIO,
    ROUND(p.MARKET_VALUE / (SELECT SUM(p2.MARKET_VALUE) FROM NEXUS_HOL.ANALYTICS.POSITIONS p2 WHERE p2.CLIENT_ID = p.CLIENT_ID) *
100, 2) AS CONCENTRATION_PCT
FROM NEXUS_HOL.ANALYTICS.POSITIONS p
JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON p.CLIENT_ID = c.CLIENT_ID
WHERE p.MARKET_VALUE / (SELECT SUM(p2.MARKET_VALUE) FROM NEXUS_HOL.ANALYTICS.POSITIONS p2 WHERE p2.CLIENT_ID = p.CLIENT_ID) > 0.10
ORDER BY CONCENTRATION_PCT DESC;

### 📌 What to look for in the EXPLAIN output:

You should see **multiple scans** of the POSITIONS table — one for each correlated subquery. That's 3 full table scans where 1 would suffice

## 🛠️ Now use CoCo to optimize it

**In Snowsight:**
1. Select the inefficient query in the cell above
2. Open **CoCo** (click the AI icon or Cmd+I)
3. Ask:

> *"Optimize this query. It's running correlated subqueries that scan the positions table multiple times. Rewrite using a window 
function."*

### ✅ CoCo's Optimized Response

CoCo should produce a single-scan rewrite using `SUM(...) OVER (PARTITION BY ...)`:

**What changed:**
- **Before:** 3 correlated subqueries = 3 full scans of POSITIONS per row
- **After:** 1 window function = 1 scan total
- **Snowflake bonus:** `QUALIFY` eliminates the need for a wrapping subquery to filter on a window function result

In [ ]:
%%sql -r OptimizedQuery
-- Optimized version (paste CoCo's output here or use this reference)
SELECT
    c.CLIENT_NAME,
    p.SYMBOL,
    p.MARKET_VALUE,
    SUM(p.MARKET_VALUE) OVER (PARTITION BY p.CLIENT_ID) AS TOTAL_PORTFOLIO,
    ROUND(p.MARKET_VALUE / SUM(p.MARKET_VALUE) OVER (PARTITION BY p.CLIENT_ID) * 100, 2) AS CONCENTRATION_PCT
FROM NEXUS_HOL.ANALYTICS.POSITIONS p
JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON p.CLIENT_ID = c.CLIENT_ID
QUALIFY CONCENTRATION_PCT > 10
ORDER BY CONCENTRATION_PCT DESC;

In [ ]:
%%sql -r ExplainOptimizedQuery
-- Compare: EXPLAIN on the optimized version
EXPLAIN
SELECT
    c.CLIENT_NAME,
    p.SYMBOL,
    p.MARKET_VALUE,
    SUM(p.MARKET_VALUE) OVER (PARTITION BY p.CLIENT_ID) AS TOTAL_PORTFOLIO,
    ROUND(p.MARKET_VALUE / SUM(p.MARKET_VALUE) OVER (PARTITION BY p.CLIENT_ID) * 100, 2) AS CONCENTRATION_PCT
FROM NEXUS_HOL.ANALYTICS.POSITIONS p
JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON p.CLIENT_ID = c.CLIENT_ID
QUALIFY CONCENTRATION_PCT > 10
ORDER BY CONCENTRATION_PCT DESC;

### 📌 Compare the two plans:

| | Inefficient (correlated subqueries) | Optimized (window function) |
|---|---|---|
| Table scans on POSITIONS | 3+ | 1 |
| Execution nodes | More (nested loops) | Fewer (single pass + partition) |
| Scales with data size | Poorly (multiplicative) | Well (linear) |

### Why this matters for agent-generated SQL:

The agent generates **correct** SQL from the semantic view — but not always **optimal** SQL. Common patterns CoCo can fix:
- Correlated subqueries → window functions
- Unnecessary DISTINCT or GROUP BY → proper joins
- Missing QUALIFY (Snowflake-specific) → subquery elimination
- Redundant CTEs → simplified single-pass queries

This is the developer's role: validate agent output, optimize what matters, and feed improvements back as verified queries so the
agent learns the better pattern.

> **Tip:** You can also click the **Query ID** in the results pane → **Profile** tab to see actual execution time and partitions
scanned side-by-side.

## Step 3: 🛠️ Validate Agent Behavior with Edge Cases

### Use CoCo to generate test questions that might trip up the agent

Open CoCo and ask:

> *"I have a Cortex Agent called NEXUS_AGENT that queries a semantic view with CLIENTS, POSITIONS, and TRADES tables. Generate 5 edge-case questions that might cause the agent to produce incorrect SQL — things like ambiguous column references, time filter confusion, or metric calculation errors."*

CoCo will generate questions like:
- "What's our total AUM including inactive clients?" (tests STATUS filter assumption)
- "Show me trades from last month" (tests time interpretation — calendar month vs 30 days?)
- "What's the average portfolio return?" (tests whether it uses unrealized PnL correctly)
- "Compare our technology exposure in EMEA vs APJ" (tests multi-filter + multi-join)
- "Which relationship manager has the worst performing clients?" (tests definition of 'worst')

### Then test each one against the agent:

Run them via `DATA_AGENT_RUN` or in the SI chat, and check if the generated SQL matches your business logic. If it doesn't → add a verified query to the semantic view (the improvement flywheel from Module 04).

## Step 4: 🔹 Other CoCo Use Cases for Agent Builders

### Beyond debugging, CoCo accelerates the entire agent development lifecycle:

| Task | How to Use CoCo |
|------|----------------|
| Generate test data | "Create 10 more sample trades for testing" |
| Write verified queries | "Write a SQL query that answers 'What's our monthly trade volume trend?'" |
| Explain agent traces | Paste a trace from Module 04's monitoring and ask "Why did the agent choose this tool?" |
| Build pipeline steps | "Write a Dynamic Table that aggregates daily trade volume from TRADES" |
| Refine semantic view | "Add a metric for 'win rate' defined as profitable trades / total trades" |
| Review SQL for errors | Select agent-generated SQL and ask "Is this correct? Check for join fanout." |

## Step 5: 🛠️ Create a Custom AI Function

### Move from interactive questions to production-grade batch AI

In Modules 03–05, the agent answers questions interactively. But what about **batch workflows** — classifying every research note, scoring every trade, extracting entities from every document?

**Custom AI Functions** let you create first-class Snowflake SQL functions that encapsulate optimized model + prompt logic into reusable, production-grade artifacts:

```sql
-- Example: A custom sentiment function with magnitude + proof phrases
SELECT
    NOTES,
    MY_CUSTOM_SENTIMENT(NOTES) AS SENTIMENT_ANALYSIS
FROM NEXUS_HOL.ANALYTICS.RESEARCH_NOTES;
```

**How it works (with AI Function Builder in CoCo):**
1. **Define** the task in natural language (via CoCo's AI Function Builder)
2. **Evaluate** across models with built-in benchmarking (cost vs quality tradeoff)
3. **Optimize** using automated prompt optimization (GEPA)
4. **Register** as a first-class Snowflake object (RBAC, observable, reusable)

**Key differentiator vs. AI_COMPLETE:** Custom AI Functions are optimized for a specific task. You get better quality at lower cost because the prompt is tuned, the model is selected based on benchmarks, and the function is governed like any other Snowflake object.

| | AI_COMPLETE | Custom AI Functions |
|---|---|---|
| **Flexibility** | Maximum — you write the prompt | Task-specific — prompt is optimized for you |
| **Optimization** | Manual iteration | Automated (model selection + prompt tuning) |
| **Governance** | None — raw function call | RBAC, observable in Account Usage |
| **Reuse** | Copy-paste prompts across teams | First-class object, shared via GRANT |
| **Cost-quality** | Unknown without manual testing | Benchmarked with Pareto tradeoff visibility |

---

### Let's build one manually: A trade conviction classifier

The AI Function Builder automates this process, but let's build one by hand so you understand the pattern underneath. We'll create a function that analyzes trade notes and classifies the trader's conviction level (HIGH, MEDIUM, LOW) with a brief rationale.

> **Documentation:** [Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql) | [AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)
>
> **Custom AI Functions (Summit 2026):** AI Function Builder available in CoCo CLI & Desktop — automates the evaluation, optimization, and registration steps below.

In [ ]:
%%sql -r CustomAIFunction
-- Create a Custom AI Function: Classify trader conviction from trade notes
-- ╔══════════════════════════════════════════════════════════════════════════════════════╗
-- ║  XXX: Replace with the AI model to use for this function.                            ║
-- ║  Hint: For a simple classification task, a smaller/faster model is often sufficient. ║
-- ║  Check available models or use 'claude-haiku-4-5' for fast/cheap classification.     ║
-- ╚══════════════════════════════════════════════════════════════════════════════════════╝
CREATE OR REPLACE FUNCTION NEXUS_HOL.ANALYTICS.CLASSIFY_TRADE_CONVICTION(trade_notes STRING)
  RETURNS VARIANT
  LANGUAGE SQL
AS
$$
  SELECT PARSE_JSON(
    REGEXP_REPLACE(
      SNOWFLAKE.CORTEX.COMPLETE(
        'XXX',
        CONCAT(
          'You are a trade analyst. Classify the trader conviction level from the following trade note. ',
          'Return ONLY a JSON object with two fields: "conviction" (HIGH, MEDIUM, or LOW) and "rationale" (one sentence 
explanation). ',
          'Do not include markdown formatting or backticks. ',
          'Trade note: ', trade_notes
        )
      ),
      '^[\\s\\S]*?\\{', '{'
    )
  )
$$;

 ### 🛠️ Test the function on real data

Now call it like any SQL function — no prompt engineering needed by the caller.

 > 📌 **Why the REGEXP_REPLACE?** LLMs sometimes wrap JSON output in markdown backticks (` ```json ... ``` `). The regex strips everything before the first `{` as a safety net. Combined with the prompt instruction "Do not include markdown formatting," this ensures the function returns clean VARIANT every time — regardless of model behavior.

In [ ]:
%%sql -r TestCustomAIFunction
-- Test: Classify conviction for recent trades
SELECT
    c.CLIENT_NAME,
    t.SYMBOL,
    t.TRADE_TYPE,
    t.NOTES,
    NEXUS_HOL.ANALYTICS.CLASSIFY_TRADE_CONVICTION(t.NOTES) AS CONVICTION_ANALYSIS
FROM NEXUS_HOL.ANALYTICS.TRADES t
JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON t.CLIENT_ID = c.CLIENT_ID
WHERE t.NOTES IS NOT NULL
LIMIT 5;

### ✅ What you just built:

A **reusable, governed AI function** that:
- Encapsulates the prompt logic (callers don't need to know how it works)
- Can be called in any SQL context — SELECT, INSERT, Dynamic Table, Task
- Is governed by RBAC (grant USAGE to control who can call it)
- Runs on the model you chose (you can swap models without changing calling code)

### Production patterns:

```sql
-- Use in a Dynamic Table for continuous scoring
CREATE OR REPLACE DYNAMIC TABLE NEXUS_HOL.ANALYTICS.TRADE_CONVICTIONS
  TARGET_LAG = '1 hour'
  WAREHOUSE = NEXUS_WH
AS
SELECT
    TRADE_ID,
    SYMBOL,
    NOTES,
    CLASSIFY_TRADE_CONVICTION(NOTES) AS CONVICTION
FROM NEXUS_HOL.ANALYTICS.TRADES
WHERE NOTES IS NOT NULL;
```

### 🧠 What Custom AI Functions (Summit 2026) will add on top:

| What you did manually | What Custom AI Functions automate |
|----------------------|----------------------------------|
| Chose the model yourself | Benchmarks multiple models for cost vs quality |
| Wrote the prompt yourself | Optimizes the prompt automatically (GEPA) |
| Tested manually | Built-in evaluation with metrics |
| Registered as a UDF | Registers as a first-class AI Function object |
| No versioning | Version tracking with performance history |

**The pattern is the same** — you're wrapping AI logic in a SQL function. Custom AI Functions just automate the optimization and
make it a governed first-class object.

| | AI_COMPLETE (raw) | Manual UDF (what you just did) | Custom AI Functions (coming) |
|---|---|---|---|
| **Reusable** | No — prompt in every call | Yes — function abstracts it | Yes — first-class object |
| **Optimized** | No | No — you picked a model | Yes — automated benchmarking |
| **Governed** | No | Basic (UDF grants) | Full (RBAC, observable, versioned) |

> **When to position Custom AI Functions:** Customer is using AI_COMPLETE for batch processing but struggling with prompt
optimization, model selection, or team standardization. They want "AI functions that just work" without the maintenance overhead.

---

## ✅ Module 07 Complete!

### You now know how to:
- Use CoCo to **debug and optimize** agent-generated SQL
- Generate **edge-case test questions** to validate agent behavior
- Use CoCo as a **developer companion** for the full agent lifecycle
- Position **Custom AI Functions** for customers moving from experimentation to production

---

### Key Takeaways:

1. **CoCo is the builder's tool; the agent is the user's tool.** They're complementary — CoCo helps you build and maintain better agents.

2. **Query optimization matters.** The agent generates correct SQL, but not always optimal SQL. CoCo closes that gap.

3. **Testing is continuous.** Use CoCo to generate edge cases, validate behavior, and feed improvements back into verified queries.

---

### Talking Points:

- "CoCo is how developers build, debug, and optimize the AI applications that business users consume through CoWork."
- "The agent writes correct SQL. CoCo makes it fast SQL."
- "Custom AI Functions close the gap between AI experimentation and production — optimized, governed, reusable task definitions as first-class Snowflake objects."

---

### Next Steps:

Continue to **Module 08: End-to-End Validation** — Run the full scenario test and DORA check to confirm your lab is complete.